# A/A for group assignment and experiment planning

Use historical data to select an assignment, decide how many candidate splits
to evaluate, and estimate the minimum detectable effect (MDE) for a fixed
cohort. Then use the saved assignment to deliver a simulated A/B intervention.

- [Generate and compare candidate assignments](#candidate-assignments)
- [Use the selected assignment for A/B](#assignment-to-ab)
- [Calculate MDE for a fixed cohort](#fixed-cohort-mde)

<a id="candidate-assignments"></a>

## 1. Generate stratified candidate assignments

A new checkout is ready to test. The eligible users and their historical
revenue are known before launch. `StratificationRole` asks HypEx to split
within each segment. Each seed produces a candidate with the requested 50/50
allocation. `best_split` reports the candidate selected by HypEx's A/A score.

The next cell uses NumPy and pandas to generate synthetic inputs. After
construction, assignments, joins and analysis use HypEx datasets.

In [1]:
import numpy as np
import pandas as pd

from hypex import AATest, ABTest
from hypex.comparators import MDEBySize
from hypex.dataset import Dataset, InfoRole, StratificationRole, TargetRole, TreatmentRole

# Synthetic data preparation (NumPy and pandas).
rng = np.random.default_rng(73)
n_users = 2_400
segment = rng.choice(["new", "returning", "loyal"], n_users, p=[0.3, 0.5, 0.2])
history = pd.DataFrame(
    {
        "user_id": np.arange(n_users),
        "segment": segment,
        "pre_revenue": rng.gamma(5, 8, n_users)
        + np.select([segment == "returning", segment == "loyal"], [15, 30]),
    }
)

# Prepare a simulated baseline outcome before allocating any variants.
history["untreated_revenue"] = (
    20 + 0.8 * history["pre_revenue"] + rng.normal(0, 12, n_users)
)

history["treated_revenue"] = history["untreated_revenue"] + 4

# HypEx input boundary. Construct a Dataset from Spark data for a Spark run.
history_roles = {
    "user_id": InfoRole(int),
    "segment": StratificationRole(str),
    "pre_revenue": TargetRole(float),
}
history_data = Dataset(data=history[list(history_roles)], roles=history_roles)
potential_outcomes = Dataset(
    data=history[["user_id", "untreated_revenue", "treated_revenue"]],
    roles={
        "user_id": InfoRole(int),
        "untreated_revenue": TargetRole(float),
        "treated_revenue": TargetRole(float),
    },
)
seeds = range(24)

fixed = AATest(
    stratification=True, n_iterations=len(seeds), random_states=seeds, dry_test=True
).execute(history_data)

  0%|          | 0/24 [00:00<?, ?it/s]

 21%|██        | 5/24 [00:00<00:00, 42.50it/s]

 42%|████▏     | 10/24 [00:00<00:00, 44.96it/s]

 62%|██████▎   | 15/24 [00:00<00:00, 45.98it/s]

 83%|████████▎ | 20/24 [00:00<00:00, 46.36it/s]

100%|██████████| 24/24 [00:00<00:00, 45.98it/s]

A fixed iteration budget gives every seed a chance and supports repeated
null diagnostics. For a quick assignment search, `early_stopping=True` lets
HypEx finish once the tests flag no differences in a candidate. A short,
adaptively stopped run provides little information about the false-positive
rate. Choose the search procedure before observing any outcomes.

In [2]:
early = AATest(
    stratification=True, n_iterations=len(seeds), random_states=seeds, early_stopping=True
).execute(history_data)

print(f"Fixed budget: {len(fixed.experiments)} candidates evaluated")
print(f"Early stopping: {len(early.experiments)} candidates evaluated")
fixed.aa_score

  0%|          | 0/24 [00:00<?, ?it/s]

  0%|          | 0/24 [00:00<?, ?it/s]

Fixed budget: 24 candidates evaluated
Early stopping: 1 candidates evaluated


,score,p-value,pass
pre_revenue TTest test_1,0.95,0.700001,True
pre_revenue KSTest test_1,0.95,NaN,True


<a id="assignment-to-ab"></a>

## 2. Deliver the variants with the selected assignment

For this example the fixed search was chosen in advance. Save its user-to-arm
mapping before exposure and check the observed allocation within each
stratum. Check that eligible users have exactly one assignment and that the IDs still match after joining.

For a live experiment, persist this mapping and use it in the delivery system.
Here we simulate delivery by adding 4 units of revenue only for assigned test
users. We then join the same mapping to the outcome log by `user_id`.

In [3]:
assignment = fixed.best_split.select(["user_id", "split"]).rename({"split": "variant"})
assignment.roles = {"user_id": InfoRole(int), "variant": TreatmentRole(str)}
# Each assignment must identify a distinct user.
assert len(assignment.select("user_id").value_counts()) == len(assignment)

assigned_history = history_data.merge(assignment, on="user_id")
# Matching row counts after the join shows that every eligible ID has an assignment.
assert len(assigned_history) == len(assignment) == len(history_data)
assigned_history.select(["segment", "variant"]).value_counts()

,segment,variant,count
0,returning,test_1,611
1,returning,control,598
2,new,test_1,368
3,new,control,347
4,loyal,test_1,241
5,loyal,control,235


In [4]:
delivery = assignment.merge(potential_outcomes, on="user_id")

# Simulate exposure using the recorded assignment, then build the readout.
control = delivery[delivery["variant"] == "control"]
treatment = delivery[delivery["variant"] == "test_1"]
control_outcomes = control.select(["user_id", "untreated_revenue"]).rename(
    {"untreated_revenue": "revenue"}
)
test_outcomes = treatment.select(["user_id", "treated_revenue"]).rename(
    {"treated_revenue": "revenue"}
)
outcomes = control_outcomes.append(test_outcomes, reset_index=True)
readout = assignment.merge(outcomes, on="user_id")

result = ABTest(equal_variance=False).execute(readout)
print("Simulated lift: 4 revenue units per user")
result.summary

Simulated lift: 4 revenue units per user


,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value
0,revenue,test_1,62.6397,66.713422,4.073722,6.503419,2.399383,5.748062,OK,0.000002


The A/B difference illustrates how the assignment reaches the estimator.
For inference that reflects this design, a randomization test under a sharp
null would generate assignments within the same strata and replay the
candidate-generation and selection procedure. The ordinary `ABTest` report
does not replay that procedure. Use the recorded assignments for the readout
even if a new run of A/A would choose different groups.
[More on rerandomization](https://arxiv.org/abs/1207.5625)

<a id="fixed-cohort-mde"></a>

## 3. Estimate MDE when traffic is fixed

Use the planned group sizes and historical revenue as a proxy for the outcome
variance. `MDEBySize` estimates the absolute mean difference detectable at
the specified significance level and power. Here the inputs are 95% confidence
(two-sided alpha of 0.05) and 80% power. This normal-approximation calculation
is a planning baseline. It does not model the precision gain from the
best-split selection or CUPED.

In [5]:
historical_metric = assigned_history.select(["pre_revenue", "variant"])
mde_result = MDEBySize.calc(
    compare_by="groups",
    target_fields_data=historical_metric.select("pre_revenue"),
    group_field_data=historical_metric.select("variant"),
    significance=0.95,
    power=0.8,
)
mde = float(next(iter(mde_result.values())).iget_values(row=0, column=0))
historical_mean = float(historical_metric.select("pre_revenue").mean())
print(f"MDE: {mde:.2f} per user ({100 * mde / historical_mean:.2f}% of historical revenue)")

MDE: 2.41 per user (4.49% of historical revenue)
